# Prediction statistics

This notebook reads the compressed trial walks generated by Notebook 05 and applies the original definitions:

- **Panel A:** bootstrap left-choice probability for iSPN-only, left-channel dSPN, and right-channel dSPN state groups.
- **Panel B:** bootstrap CLAW-style terminal probabilities for states 8 versus 8→10 and states 4 versus 4→5.
- **Panel C:** raw decision-time distributions for trials with versus without later opponent-channel iSPN recruitment.

For CBGT Panel C, the original analysis used the exact model `decisionduration` values.
The notebook therefore creates a compact exact trial-metadata table at `data/source/cbgt/trial_metadata_exact.csv.gz`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from spn_figures.config import DERIVED, SOURCE
from spn_figures.io import read_csv, write_csv
from spn_figures.prediction import (
    N_BOOTSTRAP,
    PREDICTION_RANDOM_SEED,
    analyze_fig5_predictions,
    apply_cbgt_exact_trial_metadata,
    long_bootstrap_table,
    run_fig5_significance_tests,
    standardized_fig5_summary,
)

sequence_paths = {
    "CBGT": DERIVED / "cbgt" / "trial_sequences.csv.gz",
    "IBL": DERIVED / "ibl" / "trial_sequences.csv.gz",
    "Steinmetz": DERIVED / "steinmetz" / "trial_sequences.csv.gz",
}
missing = [path for path in sequence_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run Notebook 05 first. Missing:" + chr(10) + chr(10).join(map(str, missing))
    )

out_dir = DERIVED / "combined"
out_dir.mkdir(parents=True, exist_ok=True)

## Exact CBGT decision-time metadata

In [3]:
exact_rt_path = SOURCE / "cbgt" / "trial_metadata_exact.csv.gz"

cbgt_rt_metadata = read_csv(exact_rt_path)
print(
    f"Loaded exact CBGT trial metadata: {exact_rt_path} "
    f"({len(cbgt_rt_metadata):,} trials)"
)

Loaded exact CBGT trial metadata: /Users/zhuojunyu/Desktop/CBGTPy_control/All_GitHub_Code/data/source/cbgt/trial_metadata_exact.csv.gz (15,000 trials)


## Run the analyses

In [4]:
cbgt_sequences = read_csv(sequence_paths["CBGT"])
cbgt_sequences = apply_cbgt_exact_trial_metadata(cbgt_sequences, cbgt_rt_metadata)

ibl_sequences = read_csv(sequence_paths["IBL"])
steinmetz_sequences = read_csv(sequence_paths["Steinmetz"])

# Match the source empirical notebook's collision-proof session labels.
ibl_session = ibl_sequences["session"].astype(str)
ibl_sequences["session"] = np.where(
    ibl_session.str.startswith("IBL::"), ibl_session, "IBL::" + ibl_session
)
steinmetz_session = steinmetz_sequences["session"].astype(str)
steinmetz_sequences["session"] = np.where(
    steinmetz_session.str.startswith("Steinmetz::"),
    steinmetz_session,
    "Steinmetz::" + steinmetz_session,
)
empirical_sequences = pd.concat([ibl_sequences, steinmetz_sequences], ignore_index=True)

cbgt_main, cbgt_bootstrap, cbgt_raw, cbgt_counts, cbgt_trials = analyze_fig5_predictions(
    cbgt_sequences,
    "CBGT",
)
empirical_main, empirical_bootstrap, empirical_raw, empirical_counts, empirical_trials = analyze_fig5_predictions(
    empirical_sequences,
    "IBL+Steinmetz",
)

print(f"Bootstrap settings: {N_BOOTSTRAP}")
print("CBGT trial count:", len(cbgt_trials))
print("Pooled empirical trial count:", len(empirical_trials))

Bootstrap settings: 5000
CBGT trial count: 15000
Pooled empirical trial count: 1138


## Statistical analyses

Panel A uses a one-way ANOVA followed by two Bonferroni-adjusted two-sample t tests. <br>
Panels B and C use the planned two-sample t tests.  <br>
Note that Panel C is tested on raw trial-level decision times.

In [5]:
anova_results, pairwise_tests = run_fig5_significance_tests(
    cbgt_bootstrap,
    empirical_bootstrap,
    cbgt_raw,
    empirical_raw,
)

## Save analysis outputs

In [6]:
def csv_safe_trial_table(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    if "compressed_seq" in frame.columns:
        frame["compressed_seq"] = frame["compressed_seq"].map(
            lambda sequence: " ".join(map(str, sequence))
        )
    return frame

write_csv(cbgt_main, out_dir / "cbgt_prediction_main_results.csv")
cbgt_bootstrap.to_csv(out_dir / "cbgt_prediction_bootstrap.csv.gz", index=False)
cbgt_raw.to_csv(out_dir / "cbgt_prediction_raw_decision_times.csv.gz", index=False)
write_csv(cbgt_counts, out_dir / "cbgt_prediction_group_counts.csv")
csv_safe_trial_table(cbgt_trials).to_csv(
    out_dir / "cbgt_prediction_trial_table.csv.gz", index=False
)

write_csv(empirical_main, out_dir / "empirical_prediction_main_results.csv")
empirical_bootstrap.to_csv(out_dir / "empirical_prediction_bootstrap.csv.gz", index=False)
empirical_raw.to_csv(out_dir / "empirical_prediction_raw_decision_times.csv.gz", index=False)
write_csv(empirical_counts, out_dir / "empirical_prediction_group_counts.csv")
csv_safe_trial_table(empirical_trials).to_csv(
    out_dir / "empirical_prediction_trial_table.csv.gz", index=False
)

boxplot_summary = standardized_fig5_summary(
    cbgt_main,
    empirical_main,
    cbgt_raw,
    empirical_raw,
)
bootstrap_long = long_bootstrap_table(cbgt_bootstrap, empirical_bootstrap)
raw_combined = pd.concat([cbgt_raw, empirical_raw], ignore_index=True, sort=False)

write_csv(boxplot_summary, out_dir / "prediction_boxplot_summary.csv")
bootstrap_long.to_csv(out_dir / "prediction_bootstrap.csv.gz", index=False)
raw_combined.to_csv(out_dir / "prediction_raw_decision_times.csv.gz", index=False)
write_csv(anova_results, out_dir / "prediction_significance_panel_A_one_way_anova.csv")
write_csv(pairwise_tests, out_dir / "prediction_significance_pairwise_tests.csv")
write_csv(pairwise_tests, out_dir / "prediction_tests.csv")

print("CBGT main results:")
display(cbgt_main)
print("Pooled empirical main results:")
display(empirical_main)
print("Planned significance tests:")
display(pairwise_tests)

CBGT main results:


,dataset,prediction,metric,group,observed,ci95_lower,ci95_upper,q02_5,q25,q50,q75,q97_5,n_eligible_trials,n_resampled_per_round,B,notes
0,CBGT,1_without_dSPN_iSPN_only_uncertainty,left_choice_probability,GU_visited_1_2_3,0.488290,0.468384,0.507806,0.468384,0.481655,0.488290,0.494926,0.507806,2562,2562,5000,"CI inside uncertainty band [0.45, 0.55]"
1,CBGT,1_without_dSPN_iSPN_only_uncertainty,left_choice_probability,GL_visited_8_or_10,0.878938,0.866120,0.891101,0.866120,0.874707,0.879001,0.883294,0.891101,5047,2562,5000,CI above 0.5
2,CBGT,1_without_dSPN_iSPN_only_uncertainty,left_choice_probability,GR_visited_4_or_5,0.130919,0.117877,0.144028,0.117877,0.126073,0.130757,0.135441,0.144028,5026,2562,5000,CI below 0.5
3,CBGT,2_same_channel_coactivation_terminal_phase,P(next=END | current_state=8),left_channel_8_to_10,0.526085,0.510972,0.541611,0.510972,0.521105,0.526377,0.531315,0.541611,4269,4269,5000,CLAW-style state END probability: END outgoing...
4,CBGT,2_same_channel_coactivation_terminal_phase,"P(next=END | previous_state=8, current_state=10)",left_channel_8_to_10,0.761708,0.731104,0.791948,0.731104,0.750680,0.761739,0.772134,0.791948,4269,4269,5000,denominator is direct compressed 8->10 episodes
5,CBGT,2_same_channel_coactivation_terminal_phase,P(next=END | current_state=4),right_channel_4_to_5,0.546377,0.531286,0.561807,0.531286,0.541071,0.546565,0.551716,0.561807,4281,4281,5000,CLAW-style state END probability: END outgoing...
6,CBGT,2_same_channel_coactivation_terminal_phase,"P(next=END | previous_state=4, current_state=5)",right_channel_4_to_5,0.757143,0.724490,0.787052,0.724490,0.746452,0.758087,0.768455,0.787052,4281,4281,5000,denominator is direct compressed 4->5 episodes


Pooled empirical main results:


,dataset,prediction,metric,group,observed,ci95_lower,ci95_upper,q02_5,q25,q50,q75,q97_5,n_eligible_trials,n_resampled_per_round,B,notes
0,IBL+Steinmetz,1_without_dSPN_iSPN_only_uncertainty,left_choice_probability,GU_visited_1_2_3,0.519337,0.460265,0.576159,0.460265,0.500000,0.519868,0.536424,0.576159,543,302,5000,"CI overlaps uncertainty band [0.45, 0.55]"
1,IBL+Steinmetz,1_without_dSPN_iSPN_only_uncertainty,left_choice_probability,GL_visited_8_or_10,0.594667,0.539735,0.649007,0.539735,0.576159,0.596026,0.612583,0.649007,375,302,5000,CI above 0.5
2,IBL+Steinmetz,1_without_dSPN_iSPN_only_uncertainty,left_choice_probability,GR_visited_4_or_5,0.360927,0.307947,0.413907,0.307947,0.341060,0.360927,0.377483,0.413907,302,302,5000,CI below 0.5
3,IBL+Steinmetz,2_same_channel_coactivation_terminal_phase,P(next=END | current_state=8),left_channel_8_to_10,0.193431,0.162392,0.225989,0.162392,0.182413,0.193224,0.204545,0.225989,352,352,5000,CLAW-style state END probability: END outgoing...
4,IBL+Steinmetz,2_same_channel_coactivation_terminal_phase,"P(next=END | previous_state=8, current_state=10)",left_channel_8_to_10,0.405797,0.298246,0.523077,0.298246,0.367089,0.404913,0.442857,0.523077,352,352,5000,denominator is direct compressed 8->10 episodes
5,IBL+Steinmetz,2_same_channel_coactivation_terminal_phase,P(next=END | current_state=4),right_channel_4_to_5,0.208431,0.172727,0.247032,0.172727,0.195108,0.208232,0.220994,0.247032,288,288,5000,CLAW-style state END probability: END outgoing...
6,IBL+Steinmetz,2_same_channel_coactivation_terminal_phase,"P(next=END | previous_state=4, current_state=5)",right_channel_4_to_5,0.387097,0.217362,0.571429,0.217362,0.324324,0.387097,0.451613,0.571429,288,288,5000,denominator is direct compressed 4->5 episodes


Planned significance tests:


,panel,source,comparison,group1,group2,group1_label,group2_label,test,alternative,data_level,...,n_group2,mean_group1,mean_group2,mean_difference_group2_minus_group1,t_statistic,df,p_value,p_adjusted,p_for_stars,significance
0,A,CBGT,iSPN-only vs Left d+i,i_only,left,iSPN-only,Left d+i,post hoc two-sample t test after one-way ANOVA,two-sided,bootstrap distribution,...,5000,0.488322,0.878891,0.390569,-2350.270279,9998.0,0.000000e+00,0.0,0.000000e+00,***
1,A,CBGT,iSPN-only vs Right d+i,i_only,right,iSPN-only,Right d+i,post hoc two-sample t test after one-way ANOVA,two-sided,bootstrap distribution,...,5000,0.488322,0.130832,-0.357490,2124.672870,9998.0,0.000000e+00,0.0,0.000000e+00,***
2,A,IBL+Steinmetz,iSPN-only vs Left d+i,i_only,left,iSPN-only,Left d+i,post hoc two-sample t test after one-way ANOVA,two-sided,bootstrap distribution,...,5000,0.518912,0.594281,0.075369,-132.323273,9998.0,0.000000e+00,0.0,0.000000e+00,***
3,A,IBL+Steinmetz,iSPN-only vs Right d+i,i_only,right,iSPN-only,Right d+i,post hoc two-sample t test after one-way ANOVA,two-sided,bootstrap distribution,...,5000,0.518912,0.360293,-0.158619,283.013919,9998.0,0.000000e+00,0.0,0.000000e+00,***
4,B,CBGT,state 8 vs 8→10,8,8_to_10,state 8,8→10,planned two-sample t test,two-sided,bootstrap distribution,...,5000,0.526237,0.761517,0.235280,-951.485638,9998.0,0.000000e+00,NaN,0.000000e+00,***
5,B,CBGT,state 4 vs 4→5,4,4_to_5,state 4,4→5,planned two-sample t test,two-sided,bootstrap distribution,...,5000,0.546461,0.757150,0.210689,-826.010945,9998.0,0.000000e+00,NaN,0.000000e+00,***
6,B,IBL+Steinmetz,state 8 vs 8→10,8,8_to_10,state 8,8→10,planned two-sample t test,two-sided,bootstrap distribution,...,5000,0.193572,0.406014,0.212442,-254.480376,9998.0,0.000000e+00,NaN,0.000000e+00,***
7,B,IBL+Steinmetz,state 4 vs 4→5,4,4_to_5,state 4,4→5,planned two-sample t test,two-sided,bootstrap distribution,...,5000,0.208472,0.389525,0.181053,-138.203651,9998.0,0.000000e+00,NaN,0.000000e+00,***
8,C,CBGT,Left (d+i) vs + Right i,state10,10_to_11,Left (d+i),+ Right i,planned two-sample t test,two-sided,raw reaction_time trial distribution,...,163,125.631579,170.312883,44.681304,-9.239550,1529.0,8.024048e-20,NaN,8.024048e-20,***
9,C,CBGT,Right (d+i) vs + Left i,state5,5_to_7,Right (d+i),+ Left i,planned two-sample t test,two-sided,raw reaction_time trial distribution,...,178,125.849145,183.337079,57.487934,-11.829861,1462.0,6.678875e-31,NaN,6.678875e-31,***
